# Test qwen

In [3]:
%cd /mnt/data1tb/thangcn/datnv2

/mnt/data1tb/thangcn/datnv2


In [5]:
from vllm import LLM, SamplingParams
import json
from openai import OpenAI
from langchain_openai import ChatOpenAI
from service.func_for_fc import rag_service_info, rag_product_info, rag_doctor_info, qa_medical, qa_symptom, book_appointment

INFO 04-16 20:13:18 [__init__.py:239] Automatically detected platform cuda.


2025-04-16 20:13:19,554	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/mnt/data1tb/thangcn/datnv2/service/func_for_fc.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [8]:
import dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os
from service.search_doc import hybrid_search

In [9]:
EMBED_MODEL = "nampham1106/bkcare-embedding" #os.getenv("EMBED_MODEL", "nampham1106/bkcare-embedding")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cpu'}
)


In [11]:
with open('/mnt/data1tb/thangcn/datnv2/prompts/tools.json', 'r') as f:
    function_schema = json.load(f)
    
available_functions = {tool['name']: globals()[tool['name']] 
                                for tool in function_schema}

# Cấu hình LLM với tool_choice
llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
    temperature=0.8,
    model="meta-llama/Llama-3.1-8B-Instruct",
)

In [12]:
tools = [
    {
        "type": "function",
        "function": tool
    } for tool in function_schema
]

In [13]:
tools

[{'type': 'function',
  'function': {'name': 'rag_service_info',
   'description': "Retrieves detailed information about one or more services based on the user's query.",
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The service or category for which detailed information is being queried. This can refer to one or more services.'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'rag_product_info',
   'description': "Retrieves detailed information about one or more products based on the user's query.",
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The product or item for which detailed information is being queried. This can refer to one or more products.'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'rag_doctor_info',
   'description': "Retrieves detailed information about one or more doctors based on the user'

In [49]:
messages=[{"role": "user", "content": "Để tránh bị lây nhiễm virus Ebola, cần chú ý những điều gì?"}],

In [46]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
)

In [47]:
response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[{"role": "user", "content": "Để tránh bị lây nhiễm virus Ebola, cần chú ý những điều gì?"}],
    tools=tools,
    tool_choice="auto"
)

In [48]:
response.choices[0].message.tool_calls[0].function

Function(arguments='{"query": "l\\u1ec7 nh\\u01b0n virus Ebola"}', name='qa_symptom')

In [34]:
function_name = response.choices[0].message.tool_calls[0].function.name
function_args = response.choices[0].message.tool_calls[0].function.arguments

In [37]:
function_args = function_args.encode('utf-8').decode('unicode_escape')

In [43]:
function_args = json.loads(function_args)

In [44]:
retriever = available_functions[function_name](**function_args)

In [45]:
print(retriever)

retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b62e6d79900>, search_kwargs={'k': 10}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7b62e6d799f0>, k=10)] weights=[0.5, 0.5]


# LLM

In [54]:

user_prompt = "Nước súc miệng Listerin 70ml có giá bao nhiêu?"
system_prompt = {
    'role': 'system',
    'content' : f"You are a helpful assistant with access to the following tools or function calls. Your task is to produce a sequence of tools or function calls necessary to generate response to the user utterance. Use the following tools or function calls as required:\n{function_schema}",
}

messages = [
    system_prompt,
    {'role': 'user', 'content': user_prompt}
]

response = llm.predict_messages(
    messages,
    tools=tools,
    tool_choice="auto",
)

In [55]:
response.additional_kwargs['tool_calls']

[{'id': 'chatcmpl-tool-b28c28331b3e412bb46732e45bd8a5f5',
  'function': {'arguments': '{"query": "N\\u01b0\\u01a1c s\\u01bcu m\\u00ecng Listerin 70ml"}',
   'name': 'rag_product_info'},
  'type': 'function'}]

In [57]:
tool_calls = response.additional_kwargs['tool_calls']

In [59]:
for tool in tool_calls:
    function_name = tool['function']['name']
    function_args = tool['function']['arguments']
    function_args = function_args.encode('utf-8').decode('unicode_escape')
    print(f"Function name: {function_name}")
    print(f"Function arguments: {function_args}")

Function name: rag_product_info
Function arguments: {"query": "Nươc sƼu mìng Listerin 70ml"}


In [60]:
function_args = json.loads(function_args)

In [61]:
retriever = available_functions[function_name](**function_args)

In [62]:
retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b62e41f4e20>, search_kwargs={'k': 10}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7b62db18d750>, k=10)], weights=[0.5, 0.5])